# FlashAttention on NL-DPE — the fully log-carry form (verified)

Companion to [`online-softmax-acam-mapping.ipynb`](online-softmax-acam-mapping.ipynb), which derives the
online-softmax recurrence and its ReRAM **log-domain** binding (`+ → exp → Σ` per matmul MAC).

**This notebook goes one step further.** The carried state `(m, l, o)` is kept *entirely* in log domain —
`(m, log l, log|o|, sign(o))` — so the block recurrence uses **no multipliers and no dividers at all**:
every ✕/÷ is an add/sub in log domain plus one ACAM exp/log. This is the **corrected** dataflow (it fixes
mistakes in the earlier deck diagram): α is carried as `log α` and never exponentiated on its own, the
epilogue replaces the final `o/l` with one sub + one exp, and `log|P| = s − m_new` is the free exp-input
that feeds *both* the p chain and the P·V chain.

**Claim (verified below):** the log-carry recurrence is numerically identical to regular attention —
max |diff| ≈ 2.2e-15 over 500 randomized cases (float ulp level).

## 1 · Recap — the recurrences we build on (from the companion notebook)

- **Textbook:** `p = softmax(qKᵀ)`, `O = pV` — materialises the length-`S` score vector.
- **Online:** carry `m, l, o`; per block `α = exp(m − m_new)` rescales everything already accumulated.
- **Blocked (flash):** same, with `B` keys per block; peak staging `O(B + d)` instead of `O(S)`.

The Python below is the golden reference + the blocked form (Bq = 1 query).

In [ ]:
import numpy as np

def attention_reference(q, K, V):
    """Textbook softmax attention — our golden output. q [Bq,d] or [d]; K,V [S,d]."""
    q2 = np.atleast_2d(q)
    s = q2 @ K.T                       # [Bq,S]
    p = np.exp(s - s.max(axis=1, keepdims=True))
    p /= p.sum(axis=1, keepdims=True)
    out = p @ V                        # [Bq,d]
    return out[0] if np.ndim(q) == 1 else out

def attention_flash(q, K, V, block=64):
    """Blocked online form — linear carried state (the companion notebook's version)."""
    q2 = np.atleast_2d(q)
    d, S = q2.shape[1], K.shape[0]
    m, l, o = -np.inf, 0.0, np.zeros((q2.shape[0], d))
    for t in range(0, S, block):
        Kb, Vb = K[t:t+block], V[t:t+block]
        s      = q2 @ Kb.T             # [Bq,B]
        m_new  = np.maximum(m, s.max(axis=1, keepdims=True))
        alpha  = np.exp(m - m_new)
        p      = np.exp(s - m_new)
        l      = l * alpha + p.sum(axis=1, keepdims=True)
        o      = o * alpha + p @ Vb
        m      = m_new
    out = o / l
    return out[0] if np.ndim(q) == 1 else out

rng = np.random.default_rng(0)
q = rng.standard_normal(128); K = rng.standard_normal((1000, 128)); V = rng.standard_normal((1000, 128))
ref = attention_reference(q, K, V)
print("flash == reference:", np.allclose(attention_flash(q, K, V, 64), ref))

: 

## 2 · The fully log-carry recurrence — step by step

All expensive math (`×`, `÷`) is replaced by adding or subtracting in the **log domain**.
The log-domain rule is:

> to multiply two numbers, **add their logs**; then **exp** turns the log back into the regular number.

The chip has one unit (ACAM) that does `exp` (takes a log-domain number and produces a regular number)
and `log` (takes a regular number and produces its log). Everything else is just adders and comparators
on the chiplet.

**Shapes (decode):** `Bq = 1` (one query) · `Bkv` = KV block size · `d` = head dimension.

### The pieces before the loop starts

| piece | what it is | shape | lives in |
|---|---|---|---|
| `log|Q1|, sign(Q)` | the query: its log-magnitude + a ±1 bit | `[Bq, d]` | log + bit |
| `log|Kb|, sign(K)` | one block of keys | `[Bkv, d]` | log + bit |
| `log|Vb|, sign(V)` | one block of values | `[Bkv, d]` | log + bit |
| `m` | running max: tallest score seen so far | `[Bq]` | regular number |
| `log l` | running log-total (normalizer) | `[Bq]` | log |
| `log|o|, sign(o)` | running answer so far | `[Bq, d]` | log + bit |

### Step 0 · compute the block's QKᵀ scores — `PA1 → PE1 → PX1 → PS1`

**What this does:** for every query-key pair `(i, j)`, compute `query i · key j` using the log-domain trick,
then sum across the head-dimension `d` to get the raw attention score.

*Sub-steps:*

1. **PA1 `+`**: `log|Q1| + log|Kb|` → `log|q·k|` `[Bq,Bkv,d]`. Adding the logs = multiplying the magnitudes.
   This step just runs on the chiplet's adders.
2. **PE1 `exp`**: `exp(log|q·k|)` → `|q·k|` `[Bq,Bkv,d]`. Raises e to each log, getting the actual product
   magnitude. This is the first ACAM pass.
3. **PX1 `⊕`**: `sign(Q) ⊕ sign(K)` → `sign(qk)` `[Bq,Bkv,d]`. The XOR of the two sign bits tells you
   whether the product is positive or negative. Just gates on the chiplet.
4. **PS1 `Σ_d`**: add together the `d` signed products for each query-key pair → **`s`** `[Bq,Bkv]`.
   This is the crossbar reduce. The result is the **linear** attention score for every key in the block.

$$ s = \sum_{d} \operatorname{sign}(Q_v)\operatorname{sign}(K_v)\cdot e^{\log|Q_1|_v+\log|K_b|_v} \qquad [Bq,Bkv]\ \mathrm{(regular\ numbers)} $$

The scores `s` live in the regular-number (linear) domain — that's on purpose: the next step needs to
compare them, and comparison is free in linear.

### Step 1 · find the tallest score — `RM → NM`

**What this does:** in regular (online) softmax, you need to know the maximum score to keep the numbers
from blowing up. Step 1 finds it.

1. **RM `max`**: `s` has `Bkv` scores per query row; pick the largest: `m_blk`.
2. **NM `max`**: `m_new = max(m_old, m_blk)`. Compare the block's max with the running max you've been
   carrying from earlier blocks.

These are pure comparators — the cheapest things on the chip. `m`, `m_blk`, and `m_new` are all regular
(linear) numbers.

### Step 2 · the correction factor α (but we never exponentiate it) — `LS`

**What this does:** when the max jumps up from `m` to `m_new`, everything you already accumulated in `l`
and `o` must be shrunk by a factor `α = e^{m − m_new}` (which is always ≤ 1). Instead of computing α
and then multiplying by it, we carry its **log**.

1. **LS `−`**: `logα = m − m_new`. The difference of two regular (linear) numbers happens to be **exactly**
   the log of the correction factor — because `α ≡ e^{m−m_{new}}` by definition. No exp or log unit is
   used; this is just a peripheral subtract.

$$ \log\alpha = m_{\text{old}} - m_{\text{new}} \qquad [Bq] $$

We will use `logα` later as a log addend — whenever we need `α × something`, we add `logα` to that
something's log, then exp once.

### Step 3 · the attention weights (also free) — `LP`

**What this does:** get the log of each key's unnormalized weight. Same trick as Step 2.

1. **LP `−`**: `log|P| = s − m_new`. Since the weight `p` is defined as `e^{s−m_{new}}`, the linear
   subtraction `s − m_{new}` *is* the log of `p`. No log unit is called; this is a peripheral subtract.

$$ \log|P| = s - m_{new} \qquad [Bq,Bkv] $$

The quantity `log|P|` is now the input to **two** downstream chains — the denominator (Step 4) and
the P·V matrix multiply (Step 5) — and it arrived without a single ACAM exp or log.

### Step 4 · update the running denominator `l`

**What this does:** `l` is the running total of all weights seen so far (the denominator in softmax).
When a new block arrives, we must (A) add up this block's new weights, (B) rescale the old total so it
matches the new running max, then (C) merge and put the result back into log form for carrying.

#### Chain A — this block's contribution — `EP → PS2`

1. **EP `exp`**: `p = e^{log|P|}`. Takes the `log|P|` from Step 3 and raises e to it → the actual linear
   weight for each key (always positive). Input: `log|P|` `[Bq,Bkv]` (log). Output: `p` `[Bq,Bkv]`
   (regular positive number). This is the ACAM exp engine.

2. **PS2 `Σ`**: `Σp = p_1 + p_2 + ... + p_{Bkv}`. The crossbar adds up all the weights across the block
   to get a single number per query row. Input: `p` `[Bq,Bkv]` (regular numbers). Output: `Σp` `[Bq]`
   (regular number).

**Chain A result:** $\Sigma p = \sum_{b} e^{\log|P|_b}$  (one exp per element, then sum)

#### Chain B — rescale the old denominator — `LA → EA`

3. **LA `+`**: `logα + log l`. Multiplying the old denominator `l` by the correction factor `α` means
   adding their logs. Inputs: `logα` `[Bq]` (log, from Step 2) + `log l` `[Bq]` (log, carried from
   the previous block). Output: `logα + log l` `[Bq]` (log).

4. **EA `exp`**: `α·l = e^{logα + log l}`. The ACAM raises e to the sum, converting back to a regular
   number. This is the old denominator, now corrected to live in the new max era. Input: `logα + log l`
   `[Bq]` (log). Output: `α·l` `[Bq]` (regular positive number).

**Chain B result:** $ \alpha\cdot l = e^{\log\alpha + \log l} $  (one add, then one exp, then sum)

#### Merge and re-log — `LB → LG`

5. **LB `+`**: `l′ = α·l + Σp`. The chiplet's adder combines the rescaled old denominator with this
   block's contribution. Both are now regular positive numbers — just add them. Inputs: `α·l` `[Bq]`
   (regular) + `Σp` `[Bq]` (regular). Output: `l′` `[Bq]` (regular number, always > 0).

6. **LG `log`**: `log l′ = log(l′)`. The ACAM takes the log of the sum, putting it back into log domain
   so it can be carried to the next block as `log l`. Input: `l′` `[Bq]` (regular). Output: `log l′`
   `[Bq]` (log).

The whole step in one equation:

$$ \log l' = \log\Big( \underbrace{e^{\log\alpha + \log l}}_{\alpha\cdot l} \;+\; \underbrace{\,\sum_{b} e^{\log|P|_b}\,}_{\Sigma p} \Big) $$

Step 4 uses **two** ACAM exp passes (EP + EA), **one** ACAM log pass (LG), three crossbar/peripheral
accumulations (PS2 + LB + LA), and zero multiplications.

### Step 5 · compute this block's output contribution (P·V) — `PV1 → PV2 → PV3 → PV4`

**What this does:** multiply each weight `p_b` by the corresponding value vector `V_b`, then add them up
across the block. This is the second matrix multiply in attention: `p · V`.

1. **PV1 `+`**: `log|P| + log|Vb|`. The log-magnitude of the product `p_b × V_b` is the sum of their logs.
   Since `log|P|` came free from Step 3, and `log|Vb|` arrived from memory, the chiplet just adds them.
   Inputs: `log|P|` `[Bq,Bkv]` + `log|Vb|` `[Bkv,d]`. Output: `log|p·V|` `[Bq,Bkv,d]` (log).

2. **PV2 `exp`**: `|p·v| = e^{log|p·V|}`. The ACAM raises e to get the actual product magnitude.
   Input: `log|p·V|` `[Bq,Bkv,d]`. Output: `|p·v|` `[Bq,Bkv,d]` (regular numbers).

3. **PV3 `⊕`**: attach the sign. The sign of `p` is always `+` (p is exp of a real number, so it's ≥ 0),
   so only the sign from `V` matters. `sign(p·v) = sign(V_b)`. Just one XOR gate per element.
   Input: `sign(V)` `[Bkv,d]` (bits). Output: `sign(p·v)` `[Bq,Bkv,d]` (bits).

4. **PV4 `Σ_Bkv`**: add all the signed `p·v` contributions across the block → **`o_part`**.
   Input: `|p·v|` (regular) + `sign(p·v)` (bits). Output: `o_part` `[Bq,d]` (regular numbers, signed).

$$ o_{\text{part}} = \sum_{b=1}^{Bkv} \operatorname{sign}(V_b)\cdot e^{\log|P|_b+\log|V_b|} \qquad [Bq,d] $$

One ACAM exp pass (PV2), one crossbar reduce (PV4), a few peripheral adds/XORs. No multipliers.

### Step 6 · update the running output `o` — `OA → OE → OB → OG / SN`

**What this does:** just like the denominator, the running output `o` (numerator of the softmax) must be
rescaled by `α`, then the block's contribution is added in, then the result is logged back into the
carried state.

1. **OA `+`**: `logα + log|o|`. Adding logs to prepare the multiplication `α × o`.
   Inputs: `logα` `[Bq]` + `log|o|` `[Bq,d]` (both log). Output: `logα + log|o|` `[Bq,d]` (log).

2. **OE `exp`**: `α·o = e^{logα + log|o|}`. The ACAM exp converts to regular numbers.
   Input: `logα + log|o|` `[Bq,d]`. Output: `α·o` `[Bq,d]` (regular, positive). The sign of `o` is
   carried separately and reapplied at the next step.

3. **OB `+`**: `o′ = α·o × sign(o) + o_part`. The peripheral adder reattaches the sign of the old `o`
   (scaled by α) and adds the block's contribution `o_part`.
   Inputs: `α·o` (regular) × `sign(o)` (bit) + `o_part` `[Bq,d]` (regular, signed).
   Output: `o′` `[Bq,d]` (regular, signed).

4. **OG `log`**: `log|o′| = log(|o′|)`. ACAM takes the log of the magnitude.
   Input: `|o′|` `[Bq,d]`. Output: `log|o′|` `[Bq,d]` (log).

5. **SN `sgn`**: `sign(o′) = sign(o′)`. The sign bit is just the sign of the regular sum from OB.
   Input: `o′` `[Bq,d]`. Output: `sign(o′)` `[Bq,d]` (bit).

$$ o' = e^{\log\alpha + \log|o|} \cdot \operatorname{sign}(o) + \sum_{b} e^{\log|P|_b + \log|V_b|} \operatorname{sign}(V_b) \qquad [Bq,d] $$

One ACAM exp pass (OE), one ACAM log pass (OG), a few peripheral adds and sign logic.

### Step 7 · what crosses the block boundary (the carries)

The chiplet keeps just four things between KV blocks; the FPGA never enters the loop:

| carry | stored as | shape |
|---|---|---|
| running max | `m` (regular number) | `[Bq]` |
| denominator | `log l` (log) | `[Bq]` |
| output magnitude | `log|o|` (log) | `[Bq,d]` |
| output sign | `sign(o)` (bit) | `[Bq,d]` |

### Step 8 · the final answer (once, after the last block) — `ES → EE`

**What this does:** the softmax output is `o/l`. In the log-domain scheme that becomes `sign(o) × e^{log|o| − log l}` — a single subtract and a single exp, no division.

1. **ES `−`**: `log|o| − log l` — subtract in the log domain = divide in the regular domain.
   Inputs: `log|o|` `[Bq,d]` + `log l` `[Bq]`. Output: `log|O|` `[Bq,d]` (log).

2. **EE `exp`**: `O = sign(o) × e^{log|O|}` — the ACAM produces the final regular-number result.
   Input: `log|O|` `[Bq,d]`. Output: `O` `[Bq,d]` (regular, signed).

$$ O = \operatorname{sign}(o) \cdot e^{\,\log|o| - \log l} $$

### Where the multiplies and divides went

| operation | standard form | log-domain replacement |
|---|---|---|
| `q · k` (dot product) | multiply, then add | + in log, then exp, then Σ |
| `p = e^{s−m_new}` (weight) | exp | subtract (free identity) |
| `α·l` (rescale) | multiply | + in log, then exp |
| `p · V` (weighted value) | multiply, then add | + in log, then exp, then Σ |
| `o / l` (normalize) | divide | − in log, then exp |

Every `×` and `÷` has been replaced by `+` or `−` in the log domain followed by exactly one `exp` or
`log` on the ACAM engine. The only linear-domain arithmetic is the two per-block micro-sums (`α·l + Σp`,
`α·o + o_part`) — and those are just additions. No multiplier or divider unit is ever needed.

In [ ]:
def attention_flash_logcarry(q, K, V, block=64):
    """Exact online attention — carried state ENTIRELY in log domain.

    Notation mirrors the dataflow (§3), node ids in comments:
      inputs   : logQ1, signQ = log|Q1|, sign(Q)   [Bq,d]
                 logKb, signKb = log|Kb|, sign(K)  [Bkv,d]
                 logVb, signVb = log|Vb|, sign(V)  [Bkv,d]
      state    : m (linear, comparisons only) · log l · log|o| · sign(o)
    No multipliers, no dividers: every ✕/÷ is an add/sub in log domain + an ACAM exp/log.
    """
    q2 = np.atleast_2d(q)
    d, S = q2.shape[1], K.shape[0]
    with np.errstate(divide="ignore", invalid="ignore"):
        logQ1, signQ = np.log(np.abs(q2)), np.sign(q2)      # log|Q1| [Bq,d] · sign(Q)
        logK, signK = np.log(np.abs(K)), np.sign(K)        # full cache — sliced per block
        logV, signV = np.log(np.abs(V)), np.sign(V)
        m, logl, logo, signo = -np.inf, -np.inf, -np.inf, 0.0
        for t in range(0, S, block):
            logKb, signKb = logK[t:t+block], signK[t:t+block]   # log|Kb| [Bkv,d] · sign(K)
            logVb, signVb = logV[t:t+block], signV[t:t+block]   # log|Vb| [Bkv,d] · sign(V)
            # QKᵀ : PA1(+) → PE1(exp) → PX1(⊕) → PS1(Σ_d)
            logqk = logKb[None,:,:] + logQ1[:,None,:]           # PA1  [Bq,Bkv,d]
            absqk = np.exp(logqk)                               # PE1  [Bq,Bkv,d]
            signqk = signKb[None,:,:] * signQ[:,None,:]         # PX1  [Bq,Bkv,d]
            s = (absqk * signqk).sum(axis=2)                    # PS1  s [Bq,Bkv]
            # running max: RM(max over Bkv) → NM(max with m) → LS(log α)
            m_blk = s.max(axis=1, keepdims=True)                # RM
            m_new = np.maximum(m, m_blk)                        # NM
            logalpha = m - m_new                                # LS  log α [Bq,1]
            # p / Σp : LP(−) → EP(exp) → PS2(Σ_Bkv)
            logP = s - m_new                                    # LP  log|P| [Bq,Bkv] — exp input, free
            sump = np.exp(logP).sum(axis=1, keepdims=True)      # EP+PS2  Σp [Bq,1]
            # l-update : LA(+) → EA(exp) → LB(+) → LG(log)
            logl = np.log(np.exp(logalpha + logl) + sump)       # LA+EA+LB+LG  log l′ [Bq,1]
            # P·V : PV1(+) → PV2(exp) → PV3(⊕ signV; sign P ≡ +) → PV4(Σ_Bkv)
            logpv = logP[:,:,None] + logVb[None,:,:]            # PV1  [Bq,Bkv,d]
            o_part = (np.exp(logpv) * signVb[None,:,:]).sum(axis=1)  # PV2+PV3+PV4  [Bq,d]
            # o-update : OA(+) → OE(exp) → OB(+) → OG(log) + SN(sgn)
            o_new = np.exp(logalpha + logo) * signo + o_part    # OA+OE+OB  [Bq,d]
            logo, signo = np.log(np.abs(o_new)), np.sign(o_new) # OG, SN
            m = m_new
        # epilogue : ES(−) → EE(exp) — replaces the division
        out = np.exp(logo - logl) * signo                       # ES+EE  O [Bq,d]
    return out[0] if np.ndim(q) == 1 else out

## 3 · The dataflow — per-op, per-wire shapes

One op per node, wires carry `name [shape]`; circles are the loop-carried log state
(dashed edges feed the next block). This is the graph the kernel-spec JSON will encode.

```mermaid
flowchart TB
    %% ---------- loop-carried state (circles) ----------
    m(("m [Bq] linear"))
    ll(("log l [Bq]"))
    lo(("log|o| [Bq,d]"))
    so(("sign(o)"))

    %% ---------- inputs, NL-DPE form (sign + log-magnitude) ----------
    init(["init: m=−∞ · log l=−∞ · log|o|=−∞ · sign(o)=0"]):::io
    lq(["log|Q1| [Bq,d]"]):::io
    sq(["sign(Q) [Bq,d]"]):::io
    lk(["log|Kb| [Bkv,d]"]):::io
    sk(["sign(K) [Bkv,d]"]):::io
    lv(["log|Vb| [Bkv,d]"]):::io
    sv(["sign(V) [Bkv,d]"]):::io

    %% ---------- one block body ----------
    subgraph LOOP["for t in range(0, S, Bkv) — one block body (log-domain, exact)"]
        direction TB
        %% QK^T : + → exp → ⊕ → Σ_d
        PA1["+"]:::periph
        PE1["exp"]:::acam
        PX1["⊕"]:::periph
        PS1["Σ_d"]:::xb
        %% running max & α
        RM["max"]:::periph
        NM["max"]:::periph
        LS["−"]:::periph
        LP["−"]:::periph
        %% p / Σp
        EP["exp"]:::acam
        PS2["Σ"]:::xb
        %% l-update
        LA["+"]:::periph
        EA["exp"]:::acam
        LB["+"]:::periph
        LG["log"]:::acam
        %% P·V : + → exp → ⊕ → Σ_Bkv
        PV1["+"]:::periph
        PV2["exp"]:::acam
        PV3["⊕"]:::periph
        PV4["Σ_Bkv"]:::xb
        %% o-update
        OA["+"]:::periph
        OE["exp"]:::acam
        OB["+"]:::periph
        OG["log"]:::acam
        SN["sgn"]:::periph
    end

    %% ---------- epilogue (once per token) ----------
    ES["−"]:::periph
    EE["exp"]:::acam
    OUT(["O [Bq,d]"]):::result

    %% ================= QK^T =================
    init --> m & ll & lo & so
    lq --> PA1
    lk --> PA1
    PA1 -->|"log|qk| [Bq,Bkv,d]"| PE1
    PE1 -->|"|q·k| [Bq,Bkv,d]"| PS1
    sq --> PX1
    sk --> PX1
    PX1 -->|"sign(qk) [Bq,Bkv,d]"| PS1
    PS1 -->|"s [Bq,Bkv]"| RM

    %% ================= running max & α =================
    RM -->|"m_blk [Bq]"| NM
    m --> NM
    NM -->|"m_new [Bq]"| LS
    m --> LS
    LS -->|"log α [Bq]"| LA
    LS -->|"log α [Bq]"| OA

    %% ================= p / Σp =================
    PS1 -->|"s [Bq,Bkv]"| LP
    NM -->|"m_new [Bq]"| LP
    LP -->|"log|P| [Bq,Bkv]"| EP
    EP -->|"p [Bq,Bkv]"| PS2
    PS2 -->|"Σp [Bq]"| LB

    %% ================= l-update =================
    LA -->|"logα+log l [Bq]"| EA
    EA -->|"α·l [Bq]"| LB
    LB -->|"l′ [Bq]"| LG
    LG -->|"log l′ [Bq]"| ll

    %% ================= P·V =================
    LP -->|"log|P| [Bq,Bkv]"| PV1
    lv --> PV1
    PV1 -->|"log|P·V| [Bq,Bkv,d]"| PV2
    PV2 -->|"|P·V| [Bq,Bkv,d]"| PV4
    sv --> PV3
    PV3 -->|"sign(PV) [Bq,Bkv,d]"| PV4
    PV4 -->|"o_part [Bq,d]"| OB

    %% ================= o-update =================
    lo --> OA
    OA -->|"logα+log|o| [Bq,d]"| OE
    OE -->|"α·o [Bq,d]"| OB
    so -.-> OB
    OB -->|"o′ [Bq,d]"| OG
    OB -->|"o′ [Bq,d]"| SN
    OG -->|"log|o′| [Bq,d]"| lo
    SN -->|"sign(o′) [Bq,d]"| so

    %% ================= carries =================
    NM -. "m ← m_new [Bq]" .-> m
    LG -. "log l ← log l′ [Bq]" .-> ll

    %% ================= epilogue =================
    ll --> ES
    lo --> ES
    ES -->|"log|O| [Bq,d]"| EE
    EE -->|"|O| [Bq,d]"| OUT
    so -.-> OUT

    classDef periph fill:#8e44ad,stroke:#5b2c6f,color:#fff
    classDef acam fill:#e67e22,stroke:#bf6000,color:#fff
    classDef xb fill:#c0392b,stroke:#922b21,color:#fff
    classDef io fill:#7f8c8d,stroke:#5d6d7e,color:#fff
    classDef state fill:#2c3e50,stroke:#1b2631,color:#fff
    classDef result fill:#c9a227,stroke:#a0861c,color:#fff
```

### Per-op ledger — one row per node (the spec seed)

| node | op | engine | inputs (shape) | output (shape) |
|---|---|---|---|---|
| PA1 | `+` | periph | `log|Q1| [Bq,d]` · `log|Kb| [Bkv,d]` | `log|qk| [Bq,Bkv,d]` |
| PE1 | `exp` | ACAM | `log|qk| [Bq,Bkv,d]` | `|q·k| [Bq,Bkv,d]` |
| PX1 | `⊕` | periph | `sign(Q) [Bq,d]` · `sign(K) [Bkv,d]` | `sign(qk) [Bq,Bkv,d]` |
| PS1 | `Σ_d` | xbar | `|q·k|` · `sign(qk)` | `s [Bq,Bkv]` |
| RM | `max` | periph | `s [Bq,Bkv]` | `m_blk [Bq]` |
| NM | `max` | periph | `m [Bq]` · `m_blk [Bq]` | `m_new [Bq]` |
| LS | `−` | periph | `m [Bq]` · `m_new [Bq]` | `log α [Bq]` |
| LP | `−` | periph | `s [Bq,Bkv]` · `m_new [Bq]` | `log|P| [Bq,Bkv]` |
| EP | `exp` | ACAM | `log|P| [Bq,Bkv]` | `p [Bq,Bkv]` |
| PS2 | `Σ` | xbar | `p [Bq,Bkv]` | `Σp [Bq]` |
| LA | `+` | periph | `log α [Bq]` · `log l [Bq]` | `logα+log l [Bq]` |
| EA | `exp` | ACAM | `logα+log l [Bq]` | `α·l [Bq]` |
| LB | `+` | periph | `α·l [Bq]` · `Σp [Bq]` | `l′ [Bq]` |
| LG | `log` | ACAM | `l′ [Bq]` | `log l′ [Bq]` |
| PV1 | `+` | periph | `log|P| [Bq,Bkv]` · `log|Vb| [Bkv,d]` | `log|P·V| [Bq,Bkv,d]` |
| PV2 | `exp` | ACAM | `log|P·V| [Bq,Bkv,d]` | `|P·V| [Bq,Bkv,d]` |
| PV3 | `⊕` | periph | `sign(V) [Bkv,d]` (sign P ≡ +) | `sign(PV) [Bq,Bkv,d]` |
| PV4 | `Σ_Bkv` | xbar | `|P·V|` · `sign(PV)` | `o_part [Bq,d]` |
| OA | `+` | periph | `log α [Bq]` · `log|o| [Bq,d]` | `logα+log|o| [Bq,d]` |
| OE | `exp` | ACAM | `logα+log|o| [Bq,d]` | `α·o [Bq,d]` |
| OB | `+` | periph | `α·o [Bq,d]` · `o_part [Bq,d]` | `o′ [Bq,d]` |
| OG | `log` | ACAM | `o′ [Bq,d]` | `log|o′| [Bq,d]` |
| SN | `sgn` | periph | `o′ [Bq,d]` | `sign(o′) [Bq,d]` |
| ES | `−` | periph | `log|o| [Bq,d]` · `log l [Bq]` | `log|O| [Bq,d]` |
| EE | `exp` | ACAM | `log|O| [Bq,d]` | `O [Bq,d]` |

## 4 · Numerical verification vs regular attention

Randomised gate: arbitrary shapes (`Bq` 1–3, `d ≤ 16`, `S ≤ 60`), **arbitrary block splits** `B ∈ [1, S]`,
500 trials. `attention_reference` is the textbook two-pass form (golden); we assert the log-carry
recurrence reproduces it. Max error should sit at float-ulp level (~1e-15).

In [ ]:
def verify(n_trials=500, seed=0, d_max=16, S_max=60, Bq_max=3):
    rng = np.random.default_rng(seed)
    worst, worst_meta = 0.0, None
    for _ in range(n_trials):
        d  = int(rng.integers(1, d_max + 1))
        S  = int(rng.integers(1, S_max + 1))
        Bq = int(rng.integers(1, Bq_max + 1))
        B  = int(rng.integers(1, S + 1))            # arbitrary block split
        q = rng.standard_normal((Bq, d))
        K = rng.standard_normal((S, d))
        V = rng.standard_normal((S, d))
        ref = attention_reference(q, K, V)
        got = attention_flash_logcarry(q, K, V, block=B)
        err = float(np.abs(ref - got).max())
        if err > worst:
            worst, worst_meta = err, dict(d=d, S=S, Bq=Bq, B=B)
    return worst, worst_meta

worst, meta = verify()
print(f"500 randomized cases (Bq 1-3, d<=16, S<=60, arbitrary B)")
print(f"max |diff| vs regular attention : {worst:.3e}   (worst case: {meta})")
print(f"identical at float level        : {worst < 1e-14}")

### Hand-sized trace — the same toy as the companion notebook (d=4, S=6, block=2)

In [ ]:
rng2 = np.random.default_rng(7)
d2, S2, B2 = 4, 6, 2
q2  = rng2.standard_normal(d2).round(2)
K2  = rng2.standard_normal((S2, d2)).round(2)
V2  = rng2.standard_normal((S2, d2)).round(2)

ref2  = attention_reference(q2, K2, V2)
got2  = attention_flash_logcarry(q2, K2, V2, block=B2)
print("regular attention  :", np.round(ref2, 5))
print("log-carry flash    :", np.round(got2, 5))
print("allclose           :", np.allclose(ref2, got2))

## 5 · What this buys — and what it does NOT

**Engine-visible savings vs the linear-carry form** (the companion notebook's `attention_flash_logdomain`):

| quantity | linear-carry | log-carry (this notebook) |
|---|---|---|
| carried state | `m, l, o` (linear) | `m, log l, log|o|, sign(o)` |
| α pass | 1 exp (then ×) | **none** — `log α` carried |
| rescales | `×` (α·l, α·o) | `+` in log domain |
| epilogue | `÷` (o/l) | `−` + `exp` |
| multipliers / dividers | per block `d+1` ×, 1 ÷ | **0** |

**Exp count is unchanged** — that is the honest, correct result: flash bounds the **working set**
(footprint `O(B+d)` instead of `O(S)`), it does **not** cut attention compute. Per block:
`B·d` exps (QKᵀ) + `B` exps (p) + 1 exp (l′) + `B·d` exps (P·V) + `d` exps (o′) — the two matmul passes
dominate, and they exist in regular attention too. Over the whole sequence: `≈ 2·S·d + S` exps, identical
to the non-flash path. Flash's win stays the **score-buffer footprint / fused streaming / numerical
stability** — which is exactly the thesis-supporting claim from the simulator work
([next-moves-plan](../../.claude-memory/next-moves-plan.md)).